# TASK02 中芯国际港股（00981）技术指标实验

**指标：** RSI / MACD / 布林带 / ATR  
**规范：** `task02_indicator_lab/spec/indicator_lab.spec.yaml`  
**数据源：** AkShare `stock_hk_daily`（前复权）

## 环境与依赖

- `akshare`：获取港股日线
- `pandas`：数据处理
- `matplotlib`：可视化
- `pyyaml`：读取 spec 参数

指标采用 **Wilder 平滑** 计算 RSI、ATR；MACD 使用标准 EMA；布林带使用 SMA + 2 倍标准差。

In [ ]:
import json
import sys
from datetime import datetime, timedelta
from pathlib import Path

import akshare as ak
import matplotlib.pyplot as plt
import pandas as pd
import yaml

NB_DIR = Path.cwd()
BASE = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
sys.path.insert(0, str(BASE / "scripts"))

from font_utils import apply_global_font, get_chinese_font
from indicators import add_all_indicators

SPEC_PATH = BASE / "spec" / "indicator_lab.spec.yaml"
with SPEC_PATH.open(encoding="utf-8") as f:
    spec = yaml.safe_load(f)

instrument = spec["instrument"]
fetch_cfg = spec["data_fetch"]
ind_cfg = spec["indicators"]

params = {
    "rsi_period": ind_cfg["RSI"]["parameters"]["period"],
    "macd": ind_cfg["MACD"]["parameters"],
    "boll": ind_cfg["BOLL"]["parameters"],
    "atr_period": ind_cfg["ATR"]["parameters"]["period"],
}

RAW_PATH = BASE / fetch_cfg["output"]["raw_csv"]
PROC_PATH = BASE / spec["naming"]["processed"]
FIG_DIR = BASE / "data" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"标的: {instrument['name']} ({instrument['code']}) {instrument['market']}")
print(f"Spec 版本: {spec['spec']['version']}")

In [ ]:
# S3 获取港股日线
lookback = fetch_cfg["lookback_days"]
start_dt = datetime.now() - timedelta(days=lookback)

df_raw = ak.stock_hk_daily(
    symbol=fetch_cfg["params"]["symbol"],
    adjust=fetch_cfg["params"]["adjust"],
)
df_raw["date"] = pd.to_datetime(df_raw["date"])
df_raw = df_raw[df_raw["date"] >= start_dt].sort_values("date").drop_duplicates("date").reset_index(drop=True)

print(f"共 {len(df_raw)} 个交易日: {df_raw['date'].iloc[0].date()} ~ {df_raw['date'].iloc[-1].date()}")
df_raw.tail()

In [ ]:
# S4 保存原始 CSV
RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
df_raw.assign(date=df_raw["date"].dt.strftime("%Y-%m-%d")).to_csv(
    RAW_PATH, index=False, encoding="utf-8-sig"
)
print(f"已保存: {RAW_PATH}")
df_raw.describe()[['open', 'high', 'low', 'close', 'volume']]

## 指标含义简述

| 指标 | 含义 |
|------|------|
| **RSI(14)** | 衡量涨跌强弱，>70 偏超买，<30 偏超卖 |
| **MACD(12,26,9)** | DIF/DEA 看趋势，柱看动能 |
| **布林带(20,2)** | 价格相对 20 日均值的偏离区间 |
| **ATR(14)** | 平均真实波幅，衡量波动大小（港元） |

### S6–S9 逐步计算指标

以下分别调用 `scripts/indicators.py` 中的函数，最后合并为完整 DataFrame。

In [ ]:
from indicators import compute_rsi, compute_macd, compute_bollinger, compute_atr

df = df_raw.copy()
df["rsi_14"] = compute_rsi(df["close"], params["rsi_period"])
print("RSI(14) 最新:", round(df["rsi_14"].iloc[-1], 2))
df[["date", "close", "rsi_14"]].tail(3)

In [ ]:
macd = compute_macd(
    df["close"],
    fast=params["macd"]["fast_period"],
    slow=params["macd"]["slow_period"],
    signal=params["macd"]["signal_period"],
)
df = df.join(macd)
print("MACD 最新 DIF/DEA/柱:", round(df["macd_dif"].iloc[-1], 4), round(df["macd_dea"].iloc[-1], 4), round(df["macd_hist"].iloc[-1], 4))
df[["date", "close", "macd_dif", "macd_dea", "macd_hist"]].tail(3)

In [ ]:
boll = compute_bollinger(
    df["close"],
    period=params["boll"]["period"],
    std_multiplier=params["boll"]["std_multiplier"],
)
df = df.join(boll)
print("布林带 最新 上/中/下:", round(df["boll_upper"].iloc[-1], 2), round(df["boll_mid"].iloc[-1], 2), round(df["boll_lower"].iloc[-1], 2))
df[["date", "close", "boll_upper", "boll_mid", "boll_lower"]].tail(3)

In [ ]:
df["atr_14"] = compute_atr(df["high"], df["low"], df["close"], params["atr_period"])
print("ATR(14) 最新:", round(df["atr_14"].iloc[-1], 4), "港元")

nan_report = df[[
    "rsi_14", "macd_dif", "macd_dea", "macd_hist",
    "boll_mid", "boll_upper", "boll_lower", "atr_14",
]].isna().sum()
print("\n各指标 NaN 行数（窗口期正常）:")
print(nan_report)
df.tail(3)

In [ ]:
# 校验指标列完整性（S6–S9 已逐步计算）
required = spec["schema"]["processed_daily"]["required_columns"]
missing = [c for c in required if c not in df.columns]
assert not missing, f"缺少列: {missing}"
print("指标列齐全 ✓")
df[["date", "close", "rsi_14", "macd_dif", "macd_dea", "macd_hist", "boll_mid", "atr_14"]].tail()

In [ ]:
# S10 保存 processed CSV + meta
df_out = df.copy()
df_out["date"] = df_out["date"].dt.strftime("%Y-%m-%d")
PROC_PATH.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(PROC_PATH, index=False, encoding="utf-8-sig")

latest = df_out.iloc[-1]
req_cols = spec["schema"]["processed_daily"]["required_columns"]
meta = {
    "instrument": instrument,
    "rows": len(df_out),
    "start_date": df_out["date"].iloc[0],
    "end_date": df_out["date"].iloc[-1],
    "computed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "spec_version": spec["spec"]["version"],
    "nan_counts": df_out[req_cols].isna().sum().to_dict(),
    "latest_snapshot": {
        "date": latest["date"],
        "close": round(float(latest["close"]), 2),
        "rsi_14": round(float(latest["rsi_14"]), 2),
        "macd_dif": round(float(latest["macd_dif"]), 4),
        "macd_dea": round(float(latest["macd_dea"]), 4),
        "macd_hist": round(float(latest["macd_hist"]), 4),
        "boll_mid": round(float(latest["boll_mid"]), 2),
        "boll_upper": round(float(latest["boll_upper"]), 2),
        "boll_lower": round(float(latest["boll_lower"]), 2),
        "atr_14": round(float(latest["atr_14"]), 4),
    },
}
meta_path = BASE / spec["naming"]["meta"]
meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"已保存: {PROC_PATH}")
print(json.dumps(meta["latest_snapshot"], ensure_ascii=False, indent=2))

In [ ]:
# S11 图1 — 收盘价 + 布林带
fp = apply_global_font()
fp_title = get_chinese_font(12)
dates = df['date']

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(dates, df['close'], label='收盘价', color='#1565C0', linewidth=1.4)
ax.plot(dates, df['boll_mid'], label='中轨', color='#888', linewidth=1)
ax.plot(dates, df['boll_upper'], label='上轨', color='#ef5350', linestyle='--', linewidth=0.9)
ax.plot(dates, df['boll_lower'], label='下轨', color='#26a69a', linestyle='--', linewidth=0.9)
ax.fill_between(dates, df['boll_lower'], df['boll_upper'], alpha=0.06, color='#888')
ax.set_title(ind_cfg['BOLL']['plot']['title'], fontproperties=fp_title)
ax.set_xlabel('交易日期', fontproperties=fp)
ax.set_ylabel('价格（港元）', fontproperties=fp)
ax.legend(prop=get_chinese_font(9))
ax.grid(True, linestyle='--', alpha=0.35)
plt.xticks(rotation=30)
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_boll.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# S12 图2 — RSI
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(dates, df['rsi_14'], color='#7b1fa2', linewidth=1.4)
for y, c in [(70, '#ef5350'), (50, '#888'), (30, '#26a69a')]:
    ax.axhline(y, color=c, linestyle='--', linewidth=0.8, alpha=0.7)
ax.set_ylim(0, 100)
ax.set_title(ind_cfg['RSI']['plot']['title'], fontproperties=fp_title)
ax.set_xlabel('交易日期', fontproperties=fp)
ax.set_ylabel('RSI', fontproperties=fp)
ax.grid(True, linestyle='--', alpha=0.35)
plt.xticks(rotation=30)
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_rsi.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# S13 图3 — MACD
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
ax1.plot(dates, df['macd_dif'], label='DIF', color='#1565C0', linewidth=1.2)
ax1.plot(dates, df['macd_dea'], label='DEA', color='#f5a623', linewidth=1.2)
ax1.set_title(ind_cfg['MACD']['plot']['title'], fontproperties=fp_title)
ax1.set_ylabel('MACD', fontproperties=fp)
ax1.legend(prop=get_chinese_font(9))
ax1.grid(True, linestyle='--', alpha=0.35)
colors = ['#ef5350' if v >= 0 else '#26a69a' for v in df['macd_hist']]
ax2.bar(dates, df['macd_hist'], color=colors, width=1.0, alpha=0.85)
ax2.axhline(0, color='#888', linewidth=0.8)
ax2.set_xlabel('交易日期', fontproperties=fp)
ax2.set_ylabel('柱', fontproperties=fp)
ax2.grid(True, linestyle='--', alpha=0.35)
plt.xticks(rotation=30)
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_macd.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# S14 图4 — ATR
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(dates, df['atr_14'], color='#00897b', linewidth=1.4)
ax.set_title(ind_cfg['ATR']['plot']['title'], fontproperties=fp_title)
ax.set_xlabel('交易日期', fontproperties=fp)
ax.set_ylabel('ATR（港元）', fontproperties=fp)
ax.grid(True, linestyle='--', alpha=0.35)
plt.xticks(rotation=30)
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_atr.png', dpi=180, bbox_inches='tight')
plt.show()

## 最新指标摘要与解读

运行上方单元格后，查看 `latest_snapshot` 输出。解读要点：

- **RSI**：若 >70 偏超买，<30 偏超卖；强趋势中可长期偏离 50
- **MACD**：DIF 在 DEA 上方且柱为正，短期动能偏强；金叉/死叉关注交叉点
- **布林带**：收盘价贴近上轨偏强，贴近下轨偏弱；带宽收窄预示波动可能放大
- **ATR**：数值越大波动越剧烈，可用于设定止损宽度（如 2×ATR）

> **声明：** 以上分析仅供学习，不构成投资建议。

In [ ]:
# S15 最新指标摘要与动态解读
snap = meta["latest_snapshot"]
close = snap["close"]
rsi = snap["rsi_14"]
dif, dea, hist = snap["macd_dif"], snap["macd_dea"], snap["macd_hist"]
boll_mid = snap["boll_mid"]
boll_upper, boll_lower = snap["boll_upper"], snap["boll_lower"]
atr = snap["atr_14"]

rsi_note = "偏超买" if rsi > 70 else ("偏超卖" if rsi < 30 else "中性区间")
macd_note = "动能偏强" if dif > dea and hist > 0 else ("动能偏弱" if dif < dea else "多空胶着")
if close > boll_upper:
    boll_note = "触及/突破上轨，偏强"
elif close < boll_lower:
    boll_note = "触及/跌破下轨，偏弱"
elif abs(close - boll_mid) / boll_mid < 0.02:
    boll_note = "贴近中轨"
else:
    boll_note = "高于中轨" if close > boll_mid else "低于中轨"

print(f"日期: {snap['date']}  收盘: {close} 港元\n")
print(f"RSI(14) = {rsi:.2f}  → {rsi_note}")
print(f"MACD: DIF={dif:.4f}, DEA={dea:.4f}, 柱={hist:.4f}  → {macd_note}")
print(f"布林带: 上={boll_upper:.2f} 中={boll_mid:.2f} 下={boll_lower:.2f}  → {boll_note}")
print(f"ATR(14) = {atr:.4f} 港元  → 止损参考宽度约 {2 * atr:.2f} 港元（2×ATR）")
print("\n近 5 日指标:")
print(df_out[["date", "close", "rsi_14", "macd_dif", "macd_dea", "atr_14"]].tail().to_string(index=False))
print("\n> 声明：以上分析仅供学习，不构成投资建议。")